<a href="https://colab.research.google.com/github/AdilM01/AI-Projects/blob/feature%2Fai-projects/Ensemble%20of%20Models%20Trained%20in%20Progressive%20Fine-Tuning%20Framework%20for%20Malware%20Image%20Classification%20%20%20A%20Comparative%20Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


PATHS = {

    "probs_dir"    : "/content/drive/MyDrive/MalwareExperiments/20260509_135427",
    "save_dir"     : "./ensemble_results",


    "drive_folder" : "MalwareExperiments/ensemble",
    "save_to_drive": True,
}


MANUAL_WEIGHTS = {
    "mobilenet_v2"       : 1.0,
    "resnet50"           : 1.0,
    "densenet121"        : 1.0,
    "xception"           : 1.0,
    "inception_resnetv2" : 1.0,
    "resnet101"          : 1.0,
}
# ─────────────────────────────────────────────────────────────────────────────

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score,
    balanced_accuracy_score, confusion_matrix,
    top_k_accuracy_score, roc_curve,
)

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# ENSEMBLE DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
ENSEMBLES = {
    "lightweight": ["mobilenet_v2", "resnet50",           "densenet121"],
    "heavyweight": ["xception",     "inception_resnetv2", "resnet101"],
}

PALETTE = {
    "lightweight_weighted_avg"  : "#3498db",
    "lightweight_majority_vote" : "#2ecc71",
    "lightweight_stacking"      : "#1abc9c",
    "heavyweight_weighted_avg"  : "#e74c3c",
    "heavyweight_majority_vote" : "#e67e22",
    "heavyweight_stacking"      : "#9b59b6",
}


# ─────────────────────────────────────────────────────────────────────────────
# GOOGLE DRIVE UTILITY
# ─────────────────────────────────────────────────────────────────────────────
def save_to_drive(local_dir: Path, drive_folder: str) -> None:
    if not PATHS["save_to_drive"]:
        return
    try:
        from google.colab import drive as colab_drive
        import shutil
        mount = Path("/content/drive")
        if not (mount / "MyDrive").exists():
            colab_drive.mount("/content/drive", force_remount=False)
        dest = mount / "MyDrive" / drive_folder
        dest.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in local_dir.rglob("*"):
            if f.is_file():
                tgt = dest / f.relative_to(local_dir)
                tgt.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, tgt)
                copied += 1
        print(f"  Drive: {copied} file(s) saved to MyDrive/{drive_folder}")
    except ImportError:
        print("  [Drive] Not in Colab — skipping.")
    except Exception as e:
        print(f"  [Drive] Failed: {e}")


# ─────────────────────────────────────────────────────────────────────────────
# WEIGHT LOADER  — reads results.json, no checkpoints needed
# ─────────────────────────────────────────────────────────────────────────────
def load_weights_from_json(probs_dir: Path) -> dict:
    """
    Reads test_f1 for each model from results.json saved in the run folder.
    Falls back to MANUAL_WEIGHTS for any model not found in the file.
    """
    json_path = probs_dir / "results.json"
    loaded = {}

    if json_path.exists():
        with open(json_path) as f:
            results = json.load(f)
        for model_name, metrics in results.items():
            f1 = metrics.get("test_f1", None)
            if f1 is not None:
                loaded[model_name] = float(f1)
                print(f"    {model_name:25s}  weight(F1) = {f1:.4f}  [results.json]")
    else:
        print(f"  [weights] results.json not found at {json_path}")
        print(f"            falling back to MANUAL_WEIGHTS for all models")

    # Fill any missing models from MANUAL_WEIGHTS
    all_needed = set(ENSEMBLES["lightweight"] + ENSEMBLES["heavyweight"])
    for mn in all_needed:
        if mn not in loaded:
            w = MANUAL_WEIGHTS.get(mn, 1.0)
            loaded[mn] = w
            print(f"    {mn:25s}  weight = {w:.4f}  [manual fallback]")

    return loaded


# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADER
# ─────────────────────────────────────────────────────────────────────────────
def load_model_outputs(probs_dir: Path, model_name: str):
    probs  = np.load(probs_dir / f"{model_name}_test_probs.npy")
    labels = np.load(probs_dir / f"{model_name}_test_labels.npy")
    return probs, labels


# ─────────────────────────────────────────────────────────────────────────────
# ENSEMBLE METHODS
# ─────────────────────────────────────────────────────────────────────────────
def weighted_average(probs_list: list, weights: list) -> np.ndarray:
    w = np.array(weights, dtype=float)
    w /= w.sum()
    return sum(p * wi for p, wi in zip(probs_list, w))


def majority_vote(probs_list: list) -> np.ndarray:
    preds    = np.stack([p.argmax(axis=1) for p in probs_list], axis=1)
    n, n_cls = probs_list[0].shape
    avg_soft = sum(probs_list) / len(probs_list)
    final    = np.zeros((n, n_cls))
    for i in range(n):
        votes  = np.bincount(preds[i], minlength=n_cls)
        winner = int(np.argmax(votes * 1000 + avg_soft[i]))   # tie-break by soft probs
        final[i, winner] = 1.0
    return final


def stacking(probs_list: list, labels: np.ndarray, n_splits: int = 5):
    """
    5-fold stacking with a LogisticRegression meta-learner.
    Returns (oof_probs [N, C], fitted_meta_model).
    """
    X_meta = np.concatenate(probs_list, axis=1)
    n_cls  = probs_list[0].shape[1]
    skf    = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    oof_prob = np.zeros((len(labels), n_cls))
    oof_pred = np.zeros(len(labels), dtype=int)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_meta, labels)):
        clf = LogisticRegression(max_iter=1000, C=1.0,
                                 multi_class="multinomial", solver="lbfgs",
                                 random_state=42)
        clf.fit(X_meta[tr_idx], labels[tr_idx])
        oof_prob[va_idx] = clf.predict_proba(X_meta[va_idx])
        oof_pred[va_idx] = clf.predict(X_meta[va_idx])
        print(f"      fold {fold+1}/{n_splits}  "
              f"acc={accuracy_score(labels[va_idx], oof_pred[va_idx]):.4f}")


    meta_model = LogisticRegression(max_iter=1000, C=1.0,
                                    multi_class="multinomial", solver="lbfgs",
                                    random_state=42)
    meta_model.fit(X_meta, labels)
    return oof_prob, meta_model


# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(probs: np.ndarray, labels: np.ndarray) -> dict:
    preds   = probs.argmax(axis=1)
    n_cls   = probs.shape[1]
    acc     = accuracy_score(labels, preds)
    f1      = f1_score(labels, preds, average="macro", zero_division=0)
    prec    = precision_score(labels, preds, average="macro", zero_division=0)
    rec     = recall_score(labels, preds, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(labels, preds)
    mcc     = matthews_corrcoef(labels, preds)
    kappa   = cohen_kappa_score(labels, preds)

    cm = confusion_matrix(labels, preds)
    spec_list = []
    for i in range(len(cm)):
        tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
        fp = cm[:,i].sum() - cm[i,i]
        spec_list.append(tn / (tn + fp + 1e-9))
    specificity = float(np.mean(spec_list))

    try:
        lb  = label_binarize(labels, classes=list(range(n_cls)))
        auc = roc_auc_score(lb, probs, multi_class="ovr", average="macro")
    except Exception:
        auc = 0.0
    try:
        top5 = top_k_accuracy_score(labels, probs, k=min(5, n_cls))
    except Exception:
        top5 = acc

    return {
        "accuracy"    : acc,
        "f1"          : f1,
        "precision"   : prec,
        "recall"      : rec,
        "auc"         : auc,
        "mcc"         : mcc,
        "kappa"       : kappa,
        "specificity" : specificity,
        "bal_acc"     : bal_acc,
        "top5_acc"    : top5,
        "confusion_matrix": cm,
    }


# ─────────────────────────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
def set_style():
    plt.rcParams.update({
        "figure.dpi": 150, "savefig.dpi": 300,
        "font.family": "serif", "font.size": 10,
        "axes.labelsize": 11, "axes.titlesize": 12,
        "legend.fontsize": 9,
        "axes.spines.top": False, "axes.spines.right": False,
    })
set_style()


def _tag(etype, method):
    return f"{etype}_{method}"


def plot_comparative_bars(all_metrics: dict, save_dir: Path):
    tags   = list(all_metrics.keys())
    colors = [PALETTE[t] for t in tags]
    mets   = ["accuracy", "f1", "precision", "recall", "auc"]
    titles = ["Accuracy", "Macro F1", "Precision", "Recall", "AUC"]
    fig, axes = plt.subplots(1, 5, figsize=(26, 5))
    x = np.arange(len(tags))
    for ax, met, ttl in zip(axes, mets, titles):
        vals = [all_metrics[t][met] for t in tags]
        bars = ax.bar(x, vals, color=colors, edgecolor="white", width=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels([t.replace("_", "\n") for t in tags],
                           rotation=0, ha="center", fontsize=7)
        ax.set_ylim(0, 1.1)
        ax.set_title(ttl, fontweight="bold")
        ax.grid(axis="y", alpha=0.25)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.01,
                    f"{v:.3f}", ha="center", fontsize=7)
    plt.suptitle("Ensemble Comparison — All Methods x Groups",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = save_dir / "ensemble_comparative_bars.png"
    plt.savefig(out, bbox_inches="tight"); plt.close()
    print(f"  Saved -> {out}")


def plot_confusion(cm: np.ndarray, tag_name: str,
                   class_names: list, save_dir: Path):
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    tl = class_names if len(class_names) <= 15 else []
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm_norm, annot=len(class_names)<=15, fmt=".2f",
                cmap="Blues", ax=ax, xticklabels=tl, yticklabels=tl,
                linewidths=0.4, vmin=0, vmax=1)
    ax.set_title(f"Confusion Matrix — {tag_name}", fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout()
    out = save_dir / f"ensemble_confusion_{tag_name}.png"
    plt.savefig(out, bbox_inches="tight"); plt.close()
    print(f"  Saved -> {out}")


def plot_roc(probs: np.ndarray, labels: np.ndarray,
             tag_name: str, save_dir: Path):
    n_cls   = probs.shape[1]
    lb      = label_binarize(labels, classes=list(range(n_cls)))
    all_fpr = np.linspace(0, 1, 300)
    tprs    = []
    for c in range(n_cls):
        try:
            fpr, tpr, _ = roc_curve(lb[:, c], probs[:, c])
            tprs.append(np.interp(all_fpr, fpr, tpr))
        except Exception:
            pass
    fig, ax = plt.subplots(figsize=(7, 6))
    if tprs:
        mean_tpr = np.mean(tprs, axis=0)
        try:
            auc_val = roc_auc_score(lb, probs, multi_class="ovr", average="macro")
        except Exception:
            auc_val = 0.0
        color = PALETTE.get(tag_name, "#333333")
        ax.plot(all_fpr, mean_tpr, lw=2, color=color,
                label=f"Macro AUC = {auc_val:.4f}")
        ax.fill_between(all_fpr, mean_tpr, alpha=0.08, color=color)
    ax.plot([0,1],[0,1], "k--", lw=0.8, alpha=0.5)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"ROC Curve — {tag_name}", fontweight="bold")
    ax.legend(); ax.grid(True, alpha=0.25)
    plt.tight_layout()
    out = save_dir / f"ensemble_roc_{tag_name}.png"
    plt.savefig(out, bbox_inches="tight"); plt.close()
    print(f"  Saved -> {out}")


def plot_radar(all_metrics: dict, save_dir: Path):
    metric_keys = ["accuracy", "f1", "auc", "mcc", "specificity", "bal_acc"]
    labels      = ["Accuracy", "F1", "AUC", "MCC", "Specificity", "Bal. Acc"]
    N       = len(metric_keys)
    angles  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for t, metrics in all_metrics.items():
        vals = []
        for k in metric_keys:
            v = metrics[k]
            if k == "mcc":
                v = (v + 1) / 2
            vals.append(float(np.clip(v, 0, 1)))
        vals += vals[:1]
        ax.plot(angles, vals, lw=2, color=PALETTE.get(t, "#333333"),
                label=t.replace("_", "\n"))
        ax.fill(angles, vals, alpha=0.06, color=PALETTE.get(t, "#333333"))
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title("Ensemble Radar Chart\n(MCC normalised to [0,1])",
                 fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.15), fontsize=8)
    plt.tight_layout()
    out = save_dir / "ensemble_radar.png"
    plt.savefig(out, bbox_inches="tight"); plt.close()
    print(f"  Saved -> {out}")


def plot_stacking_coef(meta_model: LogisticRegression,
                       model_names: list, n_cls: int,
                       tag_name: str, save_dir: Path):
    try:
        coef = meta_model.coef_   # [n_cls, M*C]
        M    = len(model_names)
        per_model = np.array([
            np.abs(coef[:, mi*n_cls:(mi+1)*n_cls]).mean(axis=1)
            for mi in range(M)
        ]).T   # [n_cls, M]
        fig, ax = plt.subplots(figsize=(max(8, M*2), max(6, n_cls*0.35)))
        sns.heatmap(per_model, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
                    xticklabels=model_names,
                    yticklabels=[f"cls {c}" for c in range(n_cls)],
                    linewidths=0.4)
        ax.set_title(
            f"Stacking Meta-Learner Weights — {tag_name}\n"
            "(mean |coef| per base model per output class)",
            fontweight="bold")
        ax.set_xlabel("Base Model"); ax.set_ylabel("Output Class")
        plt.tight_layout()
        out = save_dir / f"stacking_coef_{tag_name}.png"
        plt.savefig(out, bbox_inches="tight"); plt.close()
        print(f"  Saved -> {out}")
    except Exception as e:
        print(f"  [stacking coef] skipped: {e}")


def save_summary_table(all_metrics: dict, save_dir: Path):
    scalar_keys = [k for k in list(all_metrics.values())[0]
                   if k != "confusion_matrix"]
    rows = []
    for t, m in all_metrics.items():
        etype, _, method = t.partition("_")
        row = {"Tag": t, "Type": etype, "Method": method}
        for k in scalar_keys:
            row[k.replace("_", " ").title()] = f"{m[k]:.4f}"
        rows.append(row)
    df = pd.DataFrame(rows)

    accs      = np.array([m["accuracy"] for m in all_metrics.values()])
    f1s       = np.array([m["f1"]       for m in all_metrics.values()])
    mccs      = np.array([m["mcc"]      for m in all_metrics.values()])
    composite = 0.4*accs + 0.3*f1s + 0.3*((mccs+1)/2)
    best_idx  = int(np.argmax(composite))
    best_tag  = list(all_metrics.keys())[best_idx]

    df.to_csv(save_dir / "ensemble_results.csv", index=False)
    print(f"\n  CSV saved -> {save_dir / 'ensemble_results.csv'}")

    fig, ax = plt.subplots(figsize=(32, max(3, len(rows)*0.9 + 2)))
    ax.axis("off")
    tbl = ax.table(cellText=df.values, colLabels=df.columns,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.1, 1.9)
    for j in range(len(df.columns)):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
        tbl[best_idx+1, j].set_facecolor("#d5f5e3")
    ax.set_title(
        f"Ensemble Results — Best: {best_tag.upper()}  "
        f"(composite={composite[best_idx]:.4f})",
        fontsize=11, fontweight="bold", pad=12)
    plt.savefig(save_dir / "ensemble_summary_table.png",
                bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Table image saved -> {save_dir / 'ensemble_summary_table.png'}")
    return df, best_tag, composite[best_idx]


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    probs_dir = Path(PATHS["probs_dir"])
    save_dir  = Path(PATHS["save_dir"])
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"  Ensemble Evaluation")
    print(f"  Run folder -> {probs_dir}")
    print(f"  Output     -> {save_dir}")
    print(f"{'='*65}")

    # ── Load weights from results.json ────────────────────────────────────
    print("\n  Loading model weights from results.json ...")
    model_weights = load_weights_from_json(probs_dir)

    # ── Load .npy soft probability arrays ─────────────────────────────────
    all_probs  = {}
    all_labels = {}
    needed = set(ENSEMBLES["lightweight"] + ENSEMBLES["heavyweight"])

    for model_name in needed:
        print(f"\n  Loading {model_name} ...")
        try:
            probs, labels = load_model_outputs(probs_dir, model_name)
            all_probs[model_name]  = probs
            all_labels[model_name] = labels
            print(f"    probs {probs.shape}   labels {labels.shape}   "
                  f"weight={model_weights[model_name]:.4f}")
        except FileNotFoundError:
            print(f"\n  ERROR: could not find files for {model_name}")
            print(f"  Expected:")
            print(f"    {probs_dir}/{model_name}_test_probs.npy")
            print(f"    {probs_dir}/{model_name}_test_labels.npy")
            print(f"  Make sure PATHS['probs_dir'] points to the correct run folder.")
            raise

    # Sanity check — all label arrays must match
    ref_labels = list(all_labels.values())[0]
    for mn, lbl in all_labels.items():
        assert np.array_equal(ref_labels, lbl), \
            f"Label mismatch for {mn}. All .npy files must be from the same test set."

    labels      = ref_labels
    n_cls       = list(all_probs.values())[0].shape[1]
    class_names = [str(c) for c in range(n_cls)]
    print(f"\n  N_test={len(labels)}   N_classes={n_cls}")


    all_metrics = {}
    meta_models = {}

    for etype, model_list in ENSEMBLES.items():
        probs_list = [all_probs[m]     for m in model_list]
        weights    = [model_weights[m] for m in model_list]

        print(f"\n{'─'*60}")
        print(f"  {etype.upper()}")
        print(f"  Members : {model_list}")
        print(f"  Weights : {[f'{w:.4f}' for w in weights]}")
        print(f"{'─'*60}")

        # 1. Weighted Averaging ────────────────────────────────────────────
        print(f"\n  [1/3] Weighted Averaging ...")
        wa_probs   = weighted_average(probs_list, weights)
        wa_metrics = compute_metrics(wa_probs, labels)
        t = _tag(etype, "weighted_avg")
        all_metrics[t] = wa_metrics
        print(f"    Acc={wa_metrics['accuracy']:.4f}  "
              f"F1={wa_metrics['f1']:.4f}  "
              f"AUC={wa_metrics['auc']:.4f}  "
              f"MCC={wa_metrics['mcc']:.4f}")
        plot_confusion(wa_metrics["confusion_matrix"], t, class_names, save_dir)
        plot_roc(wa_probs, labels, t, save_dir)

        # 2. Majority Voting ────────────────────────────────────────────────
        print(f"\n  [2/3] Majority Voting ...")
        mv_probs   = majority_vote(probs_list)
        mv_metrics = compute_metrics(mv_probs, labels)
        t = _tag(etype, "majority_vote")
        all_metrics[t] = mv_metrics
        print(f"    Acc={mv_metrics['accuracy']:.4f}  "
              f"F1={mv_metrics['f1']:.4f}  "
              f"AUC={mv_metrics['auc']:.4f}  "
              f"MCC={mv_metrics['mcc']:.4f}")
        plot_confusion(mv_metrics["confusion_matrix"], t, class_names, save_dir)
        plot_roc(mv_probs, labels, t, save_dir)

        # 3. Stacking ───────────────────────────────────────────────────────
        print(f"\n  [3/3] Stacking (5-fold LogisticRegression meta-learner) ...")
        st_probs, meta_model = stacking(probs_list, labels, n_splits=5)
        st_metrics = compute_metrics(st_probs, labels)
        t = _tag(etype, "stacking")
        all_metrics[t] = st_metrics
        meta_models[t]  = meta_model
        print(f"    Acc={st_metrics['accuracy']:.4f}  "
              f"F1={st_metrics['f1']:.4f}  "
              f"AUC={st_metrics['auc']:.4f}  "
              f"MCC={st_metrics['mcc']:.4f}")
        plot_confusion(st_metrics["confusion_matrix"], t, class_names, save_dir)
        plot_roc(st_probs, labels, t, save_dir)
        plot_stacking_coef(meta_model, model_list, n_cls, t, save_dir)

    # ── Summary plots ─────────────────────────────────────────────────────
    print("\n  Generating summary plots ...")
    plot_comparative_bars(all_metrics, save_dir)
    plot_radar(all_metrics, save_dir)
    df, best_tag, best_score = save_summary_table(all_metrics, save_dir)

    # ── Leaderboard ───────────────────────────────────────────────────────
    scalar_keys = [k for k in list(all_metrics.values())[0]
                   if k != "confusion_matrix"]
    col_w = 10
    print(f"\n{'='*90}")
    print(f"  LEADERBOARD")
    print(f"{'='*90}")
    print(f"  {'Ensemble':<37}" +
          "".join(f"{k:>{col_w}}" for k in scalar_keys))
    print(f"  {'-'*88}")
    for t, m in all_metrics.items():
        marker = " * " if t == best_tag else "   "
        print(f"{marker}{t:<37}" +
              "".join(f"{m[k]:>{col_w}.4f}" for k in scalar_keys))
    print(f"\n  BEST: {best_tag.upper()}  (composite={best_score:.4f})")
    print(f"{'='*90}")

    # ── Drive upload ──────────────────────────────────────────────────────
    if PATHS["save_to_drive"]:
        print(f"\n  Saving to Google Drive ...")
        save_to_drive(save_dir, PATHS["drive_folder"])

    print(f"\n  All outputs -> {save_dir}")
    print(f"  Files saved:")
    for f in sorted(save_dir.iterdir()):
        print(f"    {f.name}")


if __name__ == "__main__":
    main()


  Ensemble Evaluation
  Run folder -> /content/drive/MyDrive/MalwareExperiments/20260509_135427
  Output     -> ensemble_results

  Loading model weights from results.json ...
    resnet50                   weight(F1) = 0.9502  [results.json]
    resnet101                  weight(F1) = 0.9496  [results.json]
    densenet121                weight(F1) = 0.9376  [results.json]
    mobilenet_v2               weight(F1) = 0.8531  [results.json]
    inception_resnetv2         weight(F1) = 0.9331  [results.json]
    xception                   weight(F1) = 0.9405  [results.json]

  Loading resnet50 ...
    probs (928, 25)   labels (928,)   weight=0.9502

  Loading resnet101 ...
    probs (928, 25)   labels (928,)   weight=0.9496

  Loading xception ...
    probs (928, 25)   labels (928,)   weight=0.9405

  Loading inception_resnetv2 ...
    probs (928, 25)   labels (928,)   weight=0.9331

  Loading mobilenet_v2 ...
    probs (928, 25)   labels (928,)   weight=0.8531

  Loading densenet121 ...